# 06 — Forward Pass Trace: Following Data Through Every Layer

This notebook traces a single prompt through every stage of the Qwen 2.5 0.5B forward pass:
**embed → [norm → attn → + → norm → mlp → +] × N → norm → unembed → logits**.

We inspect the residual stream, attention patterns, and component contributions at each layer to build intuition for how transformer inference actually works.

In [ ]:
import sys
sys.path.insert(0, "..")

import torch
import numpy as np
import matplotlib.pyplot as plt

from utils.model_loading import load_tlens_model
from utils.visualization import plot_attention_pattern
from utils.constants import COLORS

model = load_tlens_model()

prompt = "The capital of France is"
token_ids = model.to_tokens(prompt)
str_tokens = model.to_str_tokens(prompt)

print(f"Prompt: {prompt!r}")
print(f"Token IDs: {token_ids}")
print(f"Tokens: {str_tokens}")
print(f"Sequence length: {token_ids.shape[1]}")

## Step 1: Token Embedding

The first operation maps each token ID to a dense vector via the embedding matrix `W_E`.
Note: Qwen 2.5 uses **RoPE** (Rotary Position Embeddings), which are applied inside the attention mechanism — there is no separate positional embedding added here.

In [ ]:
with torch.no_grad():
    embeddings = model.W_E[token_ids[0]]  # shape: [seq_len, d_model]

print(f"Embedding shape: {embeddings.shape}")
print(f"d_model: {model.cfg.d_model}")

# L2 norm per token position
norms = embeddings.float().norm(dim=-1).cpu().numpy()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(str_tokens)), norms, color=COLORS["embedding"], alpha=0.8)
ax.set_xticks(range(len(str_tokens)))
ax.set_xticklabels(str_tokens, rotation=45, ha="right")
ax.set_title("Token Embedding L2 Norm per Position")
ax.set_ylabel("L2 Norm")
ax.set_xlabel("Token")
plt.tight_layout()
plt.show()

print(f"\nNorms: {norms}")
print("No positional embedding added — RoPE handles position inside attention.")

## Step 2: Run with Cache

TransformerLens lets us run the full forward pass while caching every intermediate activation. This gives us access to residual streams, attention patterns, MLP outputs, and more.

In [ ]:
with torch.no_grad():
    logits, cache = model.run_with_cache(prompt)

print(f"Logits shape: {logits.shape}")
print(f"Cache keys: {len(cache)} total\n")

# Show first 20 keys to understand the naming convention
cache_keys = list(cache.keys())
print("First 20 cache keys:")
for k in cache_keys[:20]:
    print(f"  {k:45s}  shape: {tuple(cache[k].shape)}")

print(f"\n... and {len(cache_keys) - 20} more.")
print("\nNaming convention: blocks.{layer}.{component}.hook_{activation_name}")

## Step 3: Residual Stream Growth

The residual stream is the backbone of the transformer — every layer reads from it and writes back to it. As layers add information, the L2 norm of the residual stream typically grows monotonically.

In [ ]:
n_layers = model.cfg.n_layers

# Compute mean L2 norm of residual stream at each layer (averaged over positions)
# Include embedding as "layer -1"
embed_norm = embeddings.float().norm(dim=-1).mean().item()
resid_norms = [embed_norm]

for i in range(n_layers):
    resid = cache[f"blocks.{i}.hook_resid_post"]
    norm = resid[0].float().norm(dim=-1).mean().item()
    resid_norms.append(norm)

layer_labels = ["emb"] + [str(i) for i in range(n_layers)]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(len(resid_norms)), resid_norms, marker="o", color=COLORS["residual"], linewidth=2)
ax.set_xticks(range(len(layer_labels)))
ax.set_xticklabels(layer_labels)
ax.set_title("Residual Stream L2 Norm Growth Across Layers")
ax.set_xlabel("Layer (emb = embedding, 0..N = transformer blocks)")
ax.set_ylabel("Mean L2 Norm (across positions)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Embedding norm: {embed_norm:.2f}")
print(f"Final layer norm: {resid_norms[-1]:.2f}")
print(f"Growth factor: {resid_norms[-1] / embed_norm:.1f}x")

## Step 4: Attention and MLP Contributions

Each transformer block has two sub-layers: attention and MLP. Both write to the residual stream. Here we compare the magnitude of their contributions at each layer.

In [ ]:
attn_norms = []
mlp_norms = []

for i in range(n_layers):
    # Attention output: sum across heads -> shape [batch, seq, d_model]
    attn_out = cache[f"blocks.{i}.attn.hook_result"]
    attn_norm = attn_out[0].float().norm(dim=-1).mean().item()
    attn_norms.append(attn_norm)

    mlp_out = cache[f"blocks.{i}.hook_mlp_out"]
    mlp_norm = mlp_out[0].float().norm(dim=-1).mean().item()
    mlp_norms.append(mlp_norm)

x = np.arange(n_layers)
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width / 2, attn_norms, width, label="Attention", color=COLORS["attention"], alpha=0.8)
ax.bar(x + width / 2, mlp_norms, width, label="MLP", color=COLORS["mlp"], alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([str(i) for i in range(n_layers)])
ax.set_title("Attention vs MLP Output Norm per Layer")
ax.set_xlabel("Layer")
ax.set_ylabel("Mean L2 Norm (across positions)")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## Step 5: Attention Patterns (Layer 0 vs Last Layer)

Attention patterns show how each token attends to previous tokens. Early layers often learn simple patterns (attend to previous token, attend to first token). Later layers develop more specialized, context-dependent patterns.

In [ ]:
last_layer = n_layers - 1

# Attention patterns: shape [batch, n_heads, seq, seq]
pattern_l0 = cache[f"blocks.0.attn.hook_pattern"][0, 0]        # layer 0, head 0
pattern_last = cache[f"blocks.{last_layer}.attn.hook_pattern"][0, 0]  # last layer, head 0

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

plot_attention_pattern(pattern_l0, str_tokens, title="Layer 0, Head 0", ax=ax1)
plot_attention_pattern(pattern_last, str_tokens, title=f"Layer {last_layer}, Head 0", ax=ax2)

plt.tight_layout()
plt.show()

print(f"Pattern shape per head: {pattern_l0.shape}")
print(f"Number of attention heads: {model.cfg.n_heads}")

## Step 6: The Final Prediction

After all layers, the residual stream passes through a final layer norm and the unembedding matrix to produce logits over the vocabulary. We look at the last token position to see what the model predicts comes next.

In [ ]:
with torch.no_grad():
    final_resid = cache[f"blocks.{model.cfg.n_layers - 1}.hook_resid_post"]
    normed = model.ln_final(final_resid)
    logits = model.unembed(normed)
    probs = torch.softmax(logits[0, -1], dim=-1)
    top_k = torch.topk(probs, 10)

print(f"Prompt: {prompt!r}")
print(f"Vocabulary size: {logits.shape[-1]:,}")
print(f"\nTop 10 predicted next tokens:\n")
print(f"{'Rank':<6} {'Token':<20} {'Probability':<12}")
print("-" * 38)
for i, (prob, idx) in enumerate(zip(top_k.values, top_k.indices)):
    token_str = model.to_single_str_token(idx.item())
    print(f"{i+1:<6} {token_str!r:<20} {prob.item():.4f}")

## Summary

The forward pass is:

**embed → [norm → attn → + → norm → mlp → +] × N → norm → unembed → logits**

Each layer reads from and writes to the **residual stream**. The residual stream is the central communication channel — attention and MLP sub-layers are parallel contributors that add information to it. The final layer norm + unembedding converts the accumulated residual into a probability distribution over the vocabulary.

Key observations from this trace:
- **Residual stream norm grows** as layers accumulate information
- **MLP contributions** tend to dominate over attention contributions in magnitude
- **Early attention heads** learn generic patterns; **later heads** develop context-specific ones
- The model's final prediction emerges from the composition of all layers' contributions